In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vaibhavkumar11/hindi-english-parallel-corpus")

print("Path to dataset files:", path)

100%|██████████| 112M/112M [00:03<00:00, 29.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/vaibhavkumar11/hindi-english-parallel-corpus/versions/1


In [2]:
!cp -r /root/.cache/kagglehub/datasets/vaibhavkumar11/hindi-english-parallel-corpus/versions/1 /content

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("/content/1/hindi_english_parallel.csv")

In [5]:
df.head()

,hindi,english
0,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें,Give your application an accessibility workout
1,एक्सेर्साइसर पहुंचनीयता अन्वेषक,Accerciser Accessibility Explorer
2,निचले पटल के लिए डिफोल्ट प्लग-इन खाका,The default plugin layout for the bottom panel
3,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका,The default plugin layout for the top panel
4,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...,A list of plugins that are disabled by default


In [6]:
df.describe()

,hindi,english
count,1555784,1560964
unique,983938,1015944
top,Name,Your names
freq,1093,363


In [7]:
df = df[['english', 'hindi']]
df.dropna(inplace=True)

# Use smaller subset for speed
df = df.sample(80000, random_state=42)

df['english'] = df['english'].str.lower()
df['hindi'] = df['hindi'].str.lower()

df['hindi'] = 'start ' + df['hindi'] + ' end'

print("Dataset size:", len(df))

Dataset size: 80000


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

eng_vocab_size = 20000
hin_vocab_size = 20000

eng_tokenizer = Tokenizer(num_words=eng_vocab_size, oov_token="<OOV>")
hin_tokenizer = Tokenizer(num_words=hin_vocab_size, oov_token="<OOV>")

eng_tokenizer.fit_on_texts(df['english'])
hin_tokenizer.fit_on_texts(df['hindi'])

eng_seq = eng_tokenizer.texts_to_sequences(df['english'])
hin_seq = hin_tokenizer.texts_to_sequences(df['hindi'])

In [10]:
max_eng_len = 18
max_hin_len = 18

eng_seq = pad_sequences(
    eng_seq,
    maxlen=max_eng_len,
    padding='post',
    truncating='post'
)

hin_seq = pad_sequences(
    hin_seq,
    maxlen=max_hin_len,
    padding='post',
    truncating='post'
)
import numpy as np
# Expand target dimension
hin_seq = np.expand_dims(hin_seq, -1)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, RepeatVector, TimeDistributed, Dropout
from tensorflow.keras.optimizers import Adam

embedding_dim = 256
latent_dim = 384

model = Sequential()

# Encoder
model.add(Embedding(
    input_dim=eng_vocab_size,
    output_dim=embedding_dim,
    input_shape=(max_eng_len,)
))

model.add(LSTM(latent_dim))
model.add(Dropout(0.3))

# Repeat context
model.add(RepeatVector(max_hin_len))

# Decoder
model.add(LSTM(latent_dim, return_sequences=True))
model.add(Dropout(0.3))

# Output
model.add(TimeDistributed(
    Dense(hin_vocab_size, activation='softmax')
))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 18, 256)        │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 384)            │       984,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 18, 384)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 18, 384)        │     1,181,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 18, 384)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 18, 20000)      │     7,700,000 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,985,760 (57.17 MB)

 Trainable params: 14,985,760 (57.17 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history1 = model.fit(
    eng_seq,
    hin_seq,
    batch_size=128,
    epochs=5,
    validation_split=0.1
)


Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 95s 156ms/step - accuracy: 0.4222 - loss: 4.9265 - val_accuracy: 0.4705 - val_loss: 3.9398
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 160ms/step - accuracy: 0.4824 - loss: 3.8874 - val_accuracy: 0.4852 - val_loss: 3.8027
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 159ms/step - accuracy: 0.4899 - loss: 3.7372 - val_accuracy: 0.4836 - val_loss: 3.7628
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 160ms/step - accuracy: 0.4909 - loss: 3.6724 - val_accuracy: 0.4885 - val_loss: 3.6738
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 147s 168ms/step - accuracy: 0.4960 - loss: 3.5559 - val_accuracy: 0.4896 - val_loss: 3.6295


In [13]:
history1 = model.fit(
    eng_seq,
    hin_seq,
    batch_size=128,
    epochs=5,
    validation_split=0.1
)


Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 91s 161ms/step - accuracy: 0.4975 - loss: 3.5012 - val_accuracy: 0.4877 - val_loss: 3.5958
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 161ms/step - accuracy: 0.5030 - loss: 3.4239 - val_accuracy: 0.4876 - val_loss: 3.5791
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 95s 169ms/step - accuracy: 0.5054 - loss: 3.3605 - val_accuracy: 0.4863 - val_loss: 3.5344
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 160ms/step - accuracy: 0.5079 - loss: 3.2908 - val_accuracy: 0.4885 - val_loss: 3.5273
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 95s 168ms/step - accuracy: 0.5084 - loss: 3.2453 - val_accuracy: 0.4887 - val_loss: 3.5169
